# CTD Data Extraction (Batch)

Reads all cast files from the **CTD Data** folder, applies TEOS-10 conversions and
smoothing, and writes one CSV per day into the corresponding day subfolder.

Run each cell in order using **Shift + Enter**. Edit only the **Configuration** cell.

## Setup

Run this cell first. If it raises an `ImportError`, uncomment and run the `pip install`
cell below it, then restart the kernel and re-run.

In [1]:
# Uncomment and run if any library is missing, then restart the kernel.
# !pip install numpy pandas gsw

In [2]:
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import gsw

print('Ready.')

Ready.


## Configuration

**Edit the values below, then run this cell.**

| Setting | Description |
|---|---|
| `CTD_DATA_DIR` | Folder that contains the dated day subfolders downloaded by the download notebook |
| `SURFACE_THRESHOLD` | Depth (m) below which the instrument is considered to have left the surface; rows shallower than this at the start of each file are discarded as the soak period |
| `SAMPLE_RATE_HZ` | Instrument sampling rate — 20 Hz for the AML-6 |
| `SMOOTH_WINDOW_S` | Rolling-mean window in seconds applied to all measured and derived columns; set to `0` to disable smoothing |
| `DEFAULT_LAT` / `DEFAULT_LON` | Fallback position used for TEOS-10 calculations when GPS was unavailable during a cast |
| `OUTLIER_BOUNDS` | Sensor values outside these ranges are replaced with `NaN` before any calculation |

In [3]:
# ── USER CONFIGURATION ──────────────────────────────────────────
# Folder containing the YYYY-MM-DD day subfolders.
CTD_DATA_DIR = Path.home() / 'Documents' / 'CTD Data'

# Depth (m) used to detect where each cast leaves the surface.
SURFACE_THRESHOLD = 1.0

# Instrument sampling rate (Hz).
SAMPLE_RATE_HZ = 20

# Smoothing window in seconds (0 = no smoothing).
SMOOTH_WINDOW_S = 1

# Fallback location when GPS is unavailable (Bamfield area).
DEFAULT_LAT =  48.84
DEFAULT_LON = -125.13

# Outlier bounds — values outside [lo, hi] are set to NaN.
OUTLIER_BOUNDS = {
    'Salinity':  (0,  42),
    'TempCT':    (-2, 35),
    'DO':        (0,  600),
    'pH':        (6,  9),
}
# ────────────────────────────────────────────────────────────────

_smooth_n = int(round(SMOOTH_WINDOW_S * SAMPLE_RATE_HZ))
print(f'CTD data directory : {CTD_DATA_DIR}')
print(f'Smoothing window   : {SMOOTH_WINDOW_S} s  ({_smooth_n} samples)')

CTD data directory : /Users/hjb62/Documents/CTD Data
Smoothing window   : 1 s  (20 samples)


## Discover Day Folders

Scans the CTD Data directory and lists every dated subfolder along with the
number of cast files it contains.

In [4]:
_day_dirs = sorted(
    p for p in CTD_DATA_DIR.iterdir()
    if p.is_dir() and re.fullmatch(r'\d{4}-\d{2}-\d{2}', p.name)
)

if not _day_dirs:
    print(f'No day folders found in {CTD_DATA_DIR}.')
    print('Run the download notebook first, or check that CTD_DATA_DIR is correct.')
else:
    print(f'Found {len(_day_dirs)} day folder(s):')
    for _d in _day_dirs:
        _n = len(list(_d.glob('*.aml')))
        print(f'  {_d.name}  —  {_n} file(s)')

Found 6 day folder(s):
  2026-05-04  —  8 file(s)
  2026-05-05  —  5 file(s)
  2026-05-07  —  3 file(s)
  2026-05-08  —  8 file(s)
  2026-05-09  —  11 file(s)
  2026-05-11  —  27 file(s)


## Extract and Save

For each day folder the cell below:

1. Reads every `.aml` cast file and trims it to the downcast
2. Flags outliers and replaces them with `NaN`
3. Computes TEOS-10 quantities (pressure `p`, Absolute Salinity `SA`,
   Conservative Temperature `CT`, potential density anomaly `σ₀`)
4. Applies a centred rolling mean of `SMOOTH_WINDOW_S` seconds to all
   measured and derived columns
5. Writes one CSV per day into the day subfolder

The CSV columns are:

| Column | Description |
|---|---|
| `date` | Cast date (YYYY-MM-DD, from the filename) |
| `time` | Cast start time (HH:MM:SS, from the filename) |
| `lat_gps`, `lon_gps` | GPS position recorded at cast time (`NaN` if unavailable) |
| `Depth` | Instrument depth (m, positive downward) |
| `TempCT`, `Salinity`, `DO`, `Turbidity`, `DOM`, `pH` | Measured variables (raw 20 Hz) |
| `p`, `SA`, `CT`, `sigma0` | TEOS-10 derived quantities |
| `*_sm` | Smoothed counterpart of each column above |

In [ ]:
def _conv_float(s):
    try:
        return float(str(s).strip())
    except ValueError:
        return float('nan')


def _parse_aml(filepath):
    """Return (DataFrame, lat, lon) for one AML cast file."""
    nameline = header = None
    cols = []
    lat_str = lon_str = 'nan'

    with open(filepath) as fh:
        for i, line in enumerate(fh):
            if 'MeasurementMetadata' in line:
                nameline = i
            elif 'MeasurementData' in line:
                header = i
            elif nameline is not None and i == nameline + 1:
                cols = line.strip().split('=', 1)[1].split(',')
            elif 'GPSLatitude=' in line:
                lat_str = line.split('=', 1)[1].strip()
            elif 'GPSLongitude=' in line:
                lon_str = line.split('=', 1)[1].strip()

    df = pd.read_csv(filepath, header=header, names=cols)
    return df, _conv_float(lat_str), _conv_float(lon_str)


def _trim_downcast(df, surface_threshold):
    """Return only the first downcast portion of a cast DataFrame."""
    # When depth drops more than 2 m below its running maximum the instrument
    # has started ascending — use the deepest point before that as the endpoint.
    _running_max = df['Depth'].expanding().max()
    _ascending   = (_running_max - df['Depth']) > 2.0
    if _ascending.any():
        _first_ascent = _ascending.index[_ascending.to_numpy().argmax()]
        maxdepth_idx  = df.loc[:_first_ascent, 'Depth'].idxmax()
    else:
        maxdepth_idx = df['Depth'].idxmax()
    pre_max      = df.loc[:maxdepth_idx]
    near_surface = pre_max[pre_max['Depth'] < surface_threshold]
    cast_start   = near_surface.index[-1] if len(near_surface) > 0 else pre_max.index[0]
    return df.loc[cast_start:maxdepth_idx].copy()


def _smooth(series):
    if _smooth_n > 1:
        return series.rolling(_smooth_n, center=True, min_periods=1).mean()
    return series.copy()


MEASURED_COLS = ['TempCT', 'Salinity', 'DO', 'Turbidity', 'DOM', 'pH']
TEOS_COLS     = ['p', 'SA', 'CT', 'sigma0']

for _day_dir in _day_dirs:
    _files = sorted(_day_dir.glob('*.aml'))
    if not _files:
        continue

    _cast_frames = []
    for _fp in _files:
        _cast_id = '_'.join(_fp.stem.split('_')[-2:])  # e.g. 2026-05-04_10-11-14

        _df, _lat, _lon = _parse_aml(_fp)
        _df = _trim_downcast(_df, SURFACE_THRESHOLD)

        # Flag outliers
        for _col, (_lo, _hi) in OUTLIER_BOUNDS.items():
            if _col in _df.columns:
                _df.loc[(_df[_col] < _lo) | (_df[_col] > _hi), _col] = np.nan

        # GPS — fall back to default location if unavailable
        _lat_use = _lat if not np.isnan(_lat) else DEFAULT_LAT
        _lon_use = _lon if not np.isnan(_lon) else DEFAULT_LON

        # TEOS-10 conversions
        _df['p']      = gsw.p_from_z(-_df['Depth'], _lat_use)
        _df['SA']     = gsw.SA_from_SP(_df['Salinity'], _df['p'], _lon_use, _lat_use)
        _df['CT']     = gsw.CT_from_t(_df['SA'], _df['TempCT'], _df['p'])
        _df['sigma0'] = gsw.sigma0(_df['SA'], _df['CT'])

        # Smoothed columns — applied per cast so the window never blends between casts
        _present_m = [c for c in MEASURED_COLS if c in _df.columns]
        for _col in _present_m + TEOS_COLS:
            _df[f'{_col}_sm'] = _smooth(_df[_col])

        # Assemble output columns
        _cast_date, _cast_time = _cast_id.split('_')
        _df['date']    = _cast_date
        _df['time']    = _cast_time.replace('-', ':')
        _df['lat_gps'] = _lat
        _df['lon_gps'] = _lon
        _df = _df.reset_index(drop=True)

        _out_cols = (
            ['date', 'time', 'lat_gps', 'lon_gps', 'Depth']
            + _present_m
            + TEOS_COLS
            + [c + '_sm' for c in _present_m + TEOS_COLS]
        )
        _cast_frames.append(_df[[c for c in _out_cols if c in _df.columns]])

    _day_df  = pd.concat(_cast_frames, ignore_index=True)
    _out_csv = _day_dir / f'{_day_dir.name}_extracted.csv'
    _day_df.to_csv(_out_csv, index=False, float_format='%.6f')
    print(f'{_day_dir.name}: {len(_files)} cast(s), {len(_day_df):,} rows  →  {_out_csv.name}')

print('\nDone.')

---
## Done

Each day folder now contains a `YYYY-MM-DD_extracted.csv` file. Open it with
**CTD Data Extraction Final.ipynb** or load it directly in Python with `pd.read_csv()`.